# Gold Layer — Incremental Load: Dimensions

**Purpose**: Loads and refreshes all 12 Gold dimension tables from Silver layer data.

**Run order** (this notebook comes first):
```
GOLD_SETUP.ipynb  →  GOLD_INCREMENTAL_LOAD_DIMENSIONS.ipynb  →  GOLD_INCREMENTAL_LOAD_FACTS.ipynb
```

## Dimension load strategy
| Strategy | Tables | Reason |
|---|---|---|
| **Truncate + Reload** | Small reference dims (payment method, order/payment status, loyalty tier, category, marketing channel, payment terms) | Row count < 50, fast full reload is safest |
| **SCD Type 2** | DIM_CUSTOMER, DIM_PRODUCT, DIM_CAMPAIGN | Track history of key attributes |

**DIM_DATE** and **DIM_CHANNEL** are seeded once in `GOLD_SETUP.ipynb` and not modified here.

In [ ]:
# ─────────────────────────────────────────────
# Cell 1 — Imports and Session
# ─────────────────────────────────────────────
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import (
    col, lit, current_timestamp, current_date, sql_expr,
    when, coalesce, upper, lower, trim, concat,
    row_number, max as spark_max, min as spark_min,
    count as spark_count, listagg
)
from snowflake.snowpark import Window
from snowflake.snowpark.types import IntegerType, StringType, BooleanType, DateType
from datetime import date

session = get_active_session()

GOLD_SCHEMA   = 'GOLD'
SILVER_SCHEMA = 'SILVER'
RUN_DATE      = date.today()

print(f'Session   : {session.get_current_database()}.{session.get_current_schema()}')
print(f'Run date  : {RUN_DATE}')
print(f'Gold      : {GOLD_SCHEMA}')
print(f'Silver    : {SILVER_SCHEMA}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 2 — Helper Functions
# ─────────────────────────────────────────────

def truncate_table(table_name):
    """Truncates a Gold table before a full reload."""
    session.sql(f'TRUNCATE TABLE {GOLD_SCHEMA}.{table_name}').collect()


def get_next_key(table_name, key_column):
    """
    Returns the next available surrogate key integer for a Gold table.
    Returns 1 if the table is empty.
    """
    result = session.sql(
        f'SELECT COALESCE(MAX({key_column}), 0) + 1 FROM {GOLD_SCHEMA}.{table_name}'
    ).collect()
    return int(result[0][0])


def count_table(table_name, schema=None):
    """Returns the row count of a table."""
    s = schema or GOLD_SCHEMA
    return session.table(f'{s}.{table_name}').count()


def validate(table_name, expected_min=1):
    """Prints row count and raises a warning if below expected_min."""
    cnt = count_table(table_name)
    status = '✓' if cnt >= expected_min else '[!]'
    print(f'  {status} {GOLD_SCHEMA}.{table_name}: {cnt:,} rows')
    return cnt


EXECUTION_STATS = []
print('Helper functions ready.')

---
## Reference Dimensions  (Truncate + Reload)
These small lookup-sourced tables are fully replaced on each run.

In [ ]:
# ─────────────────────────────────────────────
# Cell 3 — DIM_PAYMENT_METHOD
# Source: SILVER.LKP_PAYMENT_METHOD
# ─────────────────────────────────────────────
truncate_table('DIM_PAYMENT_METHOD')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_PAYMENT_METHOD
    (PAYMENT_METHOD_KEY, PAYMENT_METHOD_CODE, PAYMENT_METHOD_NAME, CATEGORY, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY PAYMENT_METHOD_CODE)  AS PAYMENT_METHOD_KEY,
    PAYMENT_METHOD_CODE,
    PAYMENT_METHOD_NAME,
    CATEGORY,
    TRUE                                              AS IS_ACTIVE
FROM {SILVER_SCHEMA}.LKP_PAYMENT_METHOD
''').collect()

cnt = validate('DIM_PAYMENT_METHOD')
EXECUTION_STATS.append(('DIM_PAYMENT_METHOD', cnt, 'truncate+reload'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 4 — DIM_ORDER_STATUS
# Source: SILVER.LKP_ORDER_STATUS (distinct CANONICAL_STATUS values)
# Derives STATUS_CATEGORY: Open / Closed / Cancelled
# ─────────────────────────────────────────────
truncate_table('DIM_ORDER_STATUS')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_ORDER_STATUS
    (ORDER_STATUS_KEY, STATUS_CODE, STATUS_NAME, STATUS_DESCRIPTION, STATUS_CATEGORY, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY CANONICAL_STATUS)  AS ORDER_STATUS_KEY,
    CANONICAL_STATUS                               AS STATUS_CODE,
    CANONICAL_STATUS                               AS STATUS_NAME,
    MAX(DESCRIPTION)                               AS STATUS_DESCRIPTION,
    CASE
        WHEN CANONICAL_STATUS IN ('pending', 'processing', 'shipped') THEN 'Open'
        WHEN CANONICAL_STATUS IN ('completed', 'delivered')           THEN 'Closed'
        WHEN CANONICAL_STATUS IN ('cancelled', 'returned')            THEN 'Cancelled'
        ELSE 'Unknown'
    END                                            AS STATUS_CATEGORY,
    TRUE                                           AS IS_ACTIVE
FROM {SILVER_SCHEMA}.LKP_ORDER_STATUS
GROUP BY CANONICAL_STATUS
''').collect()

cnt = validate('DIM_ORDER_STATUS')
EXECUTION_STATS.append(('DIM_ORDER_STATUS', cnt, 'truncate+reload'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 5 — DIM_PAYMENT_STATUS
# Source: SILVER.LKP_PAYMENT_STATUS
# ─────────────────────────────────────────────
truncate_table('DIM_PAYMENT_STATUS')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_PAYMENT_STATUS
    (PAYMENT_STATUS_KEY, STATUS_CODE, STATUS_NAME, IS_SUCCESSFUL, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY STATUS_CODE)  AS PAYMENT_STATUS_KEY,
    STATUS_CODE,
    STATUS_NAME,
    IS_SUCCESSFUL,
    TRUE                                      AS IS_ACTIVE
FROM {SILVER_SCHEMA}.LKP_PAYMENT_STATUS
''').collect()

cnt = validate('DIM_PAYMENT_STATUS')
EXECUTION_STATS.append(('DIM_PAYMENT_STATUS', cnt, 'truncate+reload'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 6 — DIM_LOYALTY_TIER
# Source: SILVER.LKP_LOYALTY_TIER (distinct CANONICAL_TIER values)
# ─────────────────────────────────────────────
truncate_table('DIM_LOYALTY_TIER')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_LOYALTY_TIER
    (LOYALTY_TIER_KEY, TIER_CODE, TIER_NAME, TIER_RANK, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY MIN(TIER_RANK))  AS LOYALTY_TIER_KEY,
    CANONICAL_TIER                               AS TIER_CODE,
    CANONICAL_TIER                               AS TIER_NAME,
    MIN(TIER_RANK)                               AS TIER_RANK,
    TRUE                                         AS IS_ACTIVE
FROM {SILVER_SCHEMA}.LKP_LOYALTY_TIER
GROUP BY CANONICAL_TIER
''').collect()

cnt = validate('DIM_LOYALTY_TIER')
EXECUTION_STATS.append(('DIM_LOYALTY_TIER', cnt, 'truncate+reload'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 7 — DIM_CATEGORY
# Source: SILVER.BRIDGE_CATEGORY (distinct CATEGORY_CODE + CATEGORY_NAME)
# ─────────────────────────────────────────────
truncate_table('DIM_CATEGORY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_CATEGORY
    (CATEGORY_KEY, CATEGORY_CODE, CATEGORY_NAME, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY CATEGORY_CODE)  AS CATEGORY_KEY,
    CATEGORY_CODE,
    MAX(CATEGORY_NAME)                          AS CATEGORY_NAME,
    TRUE                                        AS IS_ACTIVE
FROM {SILVER_SCHEMA}.BRIDGE_CATEGORY
GROUP BY CATEGORY_CODE
''').collect()

cnt = validate('DIM_CATEGORY')
EXECUTION_STATS.append(('DIM_CATEGORY', cnt, 'truncate+reload'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 8 — DIM_MARKETING_CHANNEL
# Source: distinct CHANNEL values from SILVER.MKTG_CAMPAIGNS
# IS_PAID = TRUE for search, display, social (paid placements)
# ─────────────────────────────────────────────
truncate_table('DIM_MARKETING_CHANNEL')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_MARKETING_CHANNEL
    (MARKETING_CHANNEL_KEY, MARKETING_CHANNEL_ID, MARKETING_CHANNEL_NAME, IS_PAID, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY CHANNEL)  AS MARKETING_CHANNEL_KEY,
    LOWER(CHANNEL)                        AS MARKETING_CHANNEL_ID,
    INITCAP(CHANNEL)                      AS MARKETING_CHANNEL_NAME,
    CASE
        WHEN LOWER(CHANNEL) IN ('search', 'display', 'social') THEN TRUE
        ELSE FALSE
    END                                   AS IS_PAID,
    TRUE                                  AS IS_ACTIVE
FROM (
    SELECT DISTINCT CHANNEL
    FROM {SILVER_SCHEMA}.MKTG_CAMPAIGNS
    WHERE CHANNEL IS NOT NULL
) channels
''').collect()

cnt = validate('DIM_MARKETING_CHANNEL')
EXECUTION_STATS.append(('DIM_MARKETING_CHANNEL', cnt, 'truncate+reload'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 9 — DIM_PAYMENT_TERMS
# Source: distinct PAYMENT_TERMS from SILVER.WHL_ORDERS
# DAYS_TO_PAY derived from term code
# ─────────────────────────────────────────────
truncate_table('DIM_PAYMENT_TERMS')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_PAYMENT_TERMS
    (PAYMENT_TERMS_KEY, PAYMENT_TERMS_CODE, PAYMENT_TERMS_NAME, DAYS_TO_PAY, IS_ACTIVE)
SELECT
    ROW_NUMBER() OVER (ORDER BY PAYMENT_TERMS)  AS PAYMENT_TERMS_KEY,
    PAYMENT_TERMS                               AS PAYMENT_TERMS_CODE,
    PAYMENT_TERMS                               AS PAYMENT_TERMS_NAME,
    CASE PAYMENT_TERMS
        WHEN 'NET30'   THEN 30
        WHEN 'NET60'   THEN 60
        WHEN 'COD'     THEN 0
        WHEN 'PREPAID' THEN 0
        ELSE NULL
    END                                         AS DAYS_TO_PAY,
    TRUE                                        AS IS_ACTIVE
FROM (
    SELECT DISTINCT PAYMENT_TERMS
    FROM {SILVER_SCHEMA}.WHL_ORDERS
    WHERE PAYMENT_TERMS IS NOT NULL
) terms
''').collect()

cnt = validate('DIM_PAYMENT_TERMS')
EXECUTION_STATS.append(('DIM_PAYMENT_TERMS', cnt, 'truncate+reload'))

---
## SCD Type 2 Dimensions

Each SCD2 load uses a **two-step pattern**:
1. **Detect changes**: Compare Silver current state to Gold current rows (`IS_CURRENT = TRUE`)
2. **Expire old versions**: `UPDATE SET VALID_TO = today, IS_CURRENT = FALSE`
3. **Insert new versions**: New rows for changed records + brand-new records

In [ ]:
# ─────────────────────────────────────────────
# Cell 10 — DIM_CUSTOMER  (SCD Type 2)
# Sources: BRIDGE_CUSTOMER_IDENTITY + WEB/MOB/WHL_CUSTOMERS + LKP_LOYALTY_TIER
#
# Step 1 — Build unified Silver customer view (latest state per canonical customer)
# Step 2 — Expire Gold rows where tracked attributes have changed
# Step 3 — Insert new customers + new versions of changed customers
# ─────────────────────────────────────────────

# Step 1: Build unified Silver customer snapshot
# Join order: BRIDGE → WEB (most complete) → MOB (limited) → WHL (B2B minimal)
# COALESCE picks first non-null value across channels
#
# FIX: BRIDGE_CUSTOMER_IDENTITY has duplicate (CANONICAL_CUSTOMER_ID, SOURCE_SYSTEM)
#      rows (e.g. multiple GUEST_xxx IDs for the same email in mobile). The old bridge
#      CTE's LEFT JOINs caused fan-out → multiple snapshot rows per customer → multiple
#      IS_CURRENT=TRUE rows in DIM_CUSTOMER.
#      Solution: Deduplicate bridge to exactly 1 row per (CANONICAL_CUSTOMER_ID, SOURCE_SYSTEM)
#      using ROW_NUMBER, picking the first non-guest SOURCE_CUSTOMER_ID when available.

silver_customers_sql = f'''
WITH bridge_ranked AS (
    SELECT
        b.CANONICAL_CUSTOMER_ID,
        b.SOURCE_SYSTEM,
        b.SOURCE_CUSTOMER_ID,
        ROW_NUMBER() OVER (
            PARTITION BY b.CANONICAL_CUSTOMER_ID, b.SOURCE_SYSTEM
            ORDER BY IFF(b.SOURCE_CUSTOMER_ID LIKE \'GUEST_%\', 1, 0),
                     b.SOURCE_CUSTOMER_ID
        ) AS rn
    FROM {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY b
    WHERE b.IS_ACTIVE = TRUE
),
bridge AS (
    SELECT CANONICAL_CUSTOMER_ID, SOURCE_SYSTEM, SOURCE_CUSTOMER_ID
    FROM bridge_ranked
    WHERE rn = 1
),
web AS (
    SELECT CUSTOMER_ID, EMAIL_ADDRESS, FIRST_NAME, LAST_NAME,
           PHONE_NUMBER, COUNTRY_CODE, CITY_NAME, POSTAL_CODE,
           CURRENCY_CODE, REGISTRATION_DATE, ACCOUNT_STATUS,
           LOYALTY_TIER, MARKETING_OPT_IN
    FROM {SILVER_SCHEMA}.WEB_CUSTOMERS
),
mob AS (
    SELECT CUSTOMER_ID, EMAIL_ADDRESS, FIRST_NAME, LAST_NAME,
           LOYALTY_TIER, COUNTRY_CODE
    FROM {SILVER_SCHEMA}.MOB_CUSTOMERS
),
whl AS (
    SELECT CUSTOMER_ID, EMAIL_ADDRESS, COMPANY_NAME
    FROM {SILVER_SCHEMA}.WHL_CUSTOMERS
),
loyalty AS (
    SELECT SOURCE_SYSTEM, SOURCE_TIER_CODE, CANONICAL_TIER, TIER_RANK
    FROM {SILVER_SCHEMA}.LKP_LOYALTY_TIER
),
-- One row per canonical customer — gather data from all channels
merged AS (
    SELECT
        all_cust.CANONICAL_CUSTOMER_ID                        AS CUSTOMER_ID,
        COALESCE(we.EMAIL_ADDRESS, mo.EMAIL_ADDRESS, wh.EMAIL_ADDRESS) AS EMAIL_ADDRESS,
        COALESCE(we.FIRST_NAME, mo.FIRST_NAME)                AS FIRST_NAME,
        COALESCE(we.LAST_NAME, mo.LAST_NAME)                  AS LAST_NAME,
        COALESCE(
            we.FIRST_NAME || \' \' || we.LAST_NAME,
            mo.FIRST_NAME || \' \' || mo.LAST_NAME
        )                                                     AS FULL_NAME,
        we.PHONE_NUMBER,
        IFF(wh.CUSTOMER_ID IS NOT NULL, \'B2B\', \'B2C\')        AS CUSTOMER_TYPE,
        wh.COMPANY_NAME,
        COALESCE(we.COUNTRY_CODE, mo.COUNTRY_CODE)            AS COUNTRY_CODE,
        we.CITY_NAME,
        we.POSTAL_CODE,
        we.CURRENCY_CODE,
        we.REGISTRATION_DATE,
        COALESCE(lw.CANONICAL_TIER, lm.CANONICAL_TIER)       AS LOYALTY_TIER,
        COALESCE(lw.TIER_RANK, lm.TIER_RANK)                 AS LOYALTY_TIER_RANK,
        we.ACCOUNT_STATUS,
        we.MARKETING_OPT_IN,
        -- Determine primary channel (prefer web > mobile > wholesale)
        CASE
            WHEN bw_web.SOURCE_CUSTOMER_ID IS NOT NULL  THEN \'web\'
            WHEN bw_mob.SOURCE_CUSTOMER_ID IS NOT NULL  THEN \'mobile\'
            ELSE \'wholesale\'
        END                                                   AS PRIMARY_CHANNEL,
        -- Build comma-separated channel list without TRIM(BOTH) which is unsupported in Snowflake
        ARRAY_TO_STRING(
            ARRAY_CONSTRUCT_COMPACT(
                IFF(bw_web.SOURCE_CUSTOMER_ID IS NOT NULL, \'web\',       NULL),
                IFF(bw_mob.SOURCE_CUSTOMER_ID IS NOT NULL, \'mobile\',    NULL),
                IFF(bw_whl.SOURCE_CUSTOMER_ID IS NOT NULL, \'wholesale\', NULL)
            ), \',\'
        )                                                     AS CHANNELS_USED
    FROM (
        SELECT DISTINCT CANONICAL_CUSTOMER_ID
        FROM {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY
        WHERE IS_ACTIVE = TRUE
    ) all_cust
    -- Bridge rows per channel (deduplicated — exactly 1 row per canonical+system)
    LEFT JOIN bridge bw_web ON bw_web.CANONICAL_CUSTOMER_ID = all_cust.CANONICAL_CUSTOMER_ID
                            AND bw_web.SOURCE_SYSTEM = \'web\'
    LEFT JOIN bridge bw_mob ON bw_mob.CANONICAL_CUSTOMER_ID = all_cust.CANONICAL_CUSTOMER_ID
                            AND bw_mob.SOURCE_SYSTEM = \'mobile\'
    LEFT JOIN bridge bw_whl ON bw_whl.CANONICAL_CUSTOMER_ID = all_cust.CANONICAL_CUSTOMER_ID
                            AND bw_whl.SOURCE_SYSTEM = \'wholesale\'
    -- Customer detail per channel
    LEFT JOIN web   we      ON we.CUSTOMER_ID              = bw_web.SOURCE_CUSTOMER_ID
    LEFT JOIN mob   mo      ON mo.CUSTOMER_ID              = bw_mob.SOURCE_CUSTOMER_ID
    LEFT JOIN whl   wh      ON wh.CUSTOMER_ID              = bw_whl.SOURCE_CUSTOMER_ID
    -- Loyalty tier canonical mapping (prefer web then mobile tier)
    LEFT JOIN loyalty lw    ON lw.SOURCE_SYSTEM            = \'web\'
                            AND lw.SOURCE_TIER_CODE        = we.LOYALTY_TIER
    LEFT JOIN loyalty lm    ON lm.SOURCE_SYSTEM            = \'mobile\'
                            AND lm.SOURCE_TIER_CODE        = mo.LOYALTY_TIER
)
SELECT * FROM merged
'''

silver_cust = session.sql(silver_customers_sql)
silver_count = silver_cust.count()
print(f'Silver unified customer snapshot: {silver_count:,} customers')

In [ ]:
# ─────────────────────────────────────────────
# Cell 11 — DIM_CUSTOMER  Step 2: Expire stale rows
# Tracked attributes: LOYALTY_TIER, ACCOUNT_STATUS, COUNTRY_CODE, EMAIL_ADDRESS
# ─────────────────────────────────────────────

# Create Silver snapshot as a temp table to avoid re-running the complex CTE twice
session.sql(f'CREATE OR REPLACE TEMP TABLE SILVER_CUST_SNAPSHOT AS {silver_customers_sql}').collect()

expire_result = session.sql(f'''
UPDATE {GOLD_SCHEMA}.DIM_CUSTOMER tgt
SET
    VALID_TO   = CURRENT_DATE(),
    IS_CURRENT = FALSE
FROM SILVER_CUST_SNAPSHOT src
WHERE tgt.CUSTOMER_ID = src.CUSTOMER_ID
  AND tgt.IS_CURRENT  = TRUE
  AND (
      COALESCE(tgt.LOYALTY_TIER, '')   != COALESCE(src.LOYALTY_TIER, '')
   OR COALESCE(tgt.ACCOUNT_STATUS, '') != COALESCE(src.ACCOUNT_STATUS, '')
   OR COALESCE(tgt.COUNTRY_CODE, '')   != COALESCE(src.COUNTRY_CODE, '')
   OR COALESCE(tgt.EMAIL_ADDRESS, '')  != COALESCE(src.EMAIL_ADDRESS, '')
  )
''').collect()

rows_expired = expire_result[0][0] if expire_result else 0
print(f'  Rows expired (changed attributes): {rows_expired:,}')

In [ ]:
# ─────────────────────────────────────────────
# Cell 12 — DIM_CUSTOMER  Step 3: Insert new + re-versioned rows
# Inserts:
#   a) Brand-new customers (CUSTOMER_ID not yet in Gold)
#   b) New versions of just-expired customers (changed attributes)
# ─────────────────────────────────────────────
next_key = get_next_key('DIM_CUSTOMER', 'CUSTOMER_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_CUSTOMER (
    CUSTOMER_KEY, CUSTOMER_ID, EMAIL_ADDRESS, FIRST_NAME, LAST_NAME, FULL_NAME,
    PHONE_NUMBER, CUSTOMER_TYPE, COMPANY_NAME, COUNTRY_CODE, CITY_NAME,
    POSTAL_CODE, CURRENCY_CODE, REGISTRATION_DATE, LOYALTY_TIER, LOYALTY_TIER_RANK,
    ACCOUNT_STATUS, MARKETING_OPT_IN, PRIMARY_CHANNEL, CHANNELS_USED,
    EFFECTIVE_DATE, VALID_TO, IS_CURRENT
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY src.CUSTOMER_ID)  AS CUSTOMER_KEY,
    src.CUSTOMER_ID,
    src.EMAIL_ADDRESS,
    src.FIRST_NAME,
    src.LAST_NAME,
    src.FULL_NAME,
    src.PHONE_NUMBER,
    src.CUSTOMER_TYPE,
    src.COMPANY_NAME,
    src.COUNTRY_CODE,
    src.CITY_NAME,
    src.POSTAL_CODE,
    src.CURRENCY_CODE,
    src.REGISTRATION_DATE,
    src.LOYALTY_TIER,
    src.LOYALTY_TIER_RANK,
    src.ACCOUNT_STATUS,
    src.MARKETING_OPT_IN,
    src.PRIMARY_CHANNEL,
    src.CHANNELS_USED,
    CURRENT_DATE()  AS EFFECTIVE_DATE,
    NULL            AS VALID_TO,
    TRUE            AS IS_CURRENT
FROM SILVER_CUST_SNAPSHOT src
WHERE NOT EXISTS (
    SELECT 1
    FROM {GOLD_SCHEMA}.DIM_CUSTOMER g
    WHERE g.CUSTOMER_ID = src.CUSTOMER_ID
      AND g.IS_CURRENT  = TRUE
)
''').collect()

cnt = validate('DIM_CUSTOMER')
EXECUTION_STATS.append(('DIM_CUSTOMER', cnt, 'SCD2'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 13 — DIM_PRODUCT  (SCD Type 2)
# Sources: WEB/MOB/WHL_ORDER_ITEMS + MKT_ORDERS + BRIDGE_CATEGORY
# SCD2 tracks: PRODUCT_STATUS changes only
# Price stats (CURRENT/AVG/MIN/MAX) are Type 1 — overwritten each load
# ─────────────────────────────────────────────

silver_products_sql = f'''
WITH all_items AS (
    SELECT PRODUCT_SKU, PRODUCT_NAME, CATEGORY_CODE, UNIT_PRICE, ORDER_DATE AS SALE_DATE
    FROM {SILVER_SCHEMA}.WEB_ORDER_ITEMS wi
    JOIN {SILVER_SCHEMA}.WEB_ORDERS wo ON wo.ORDER_ID = wi.ORDER_ID

    UNION ALL

    SELECT PRODUCT_SKU, PRODUCT_NAME, CATEGORY_CODE, UNIT_PRICE, ORDER_DATE AS SALE_DATE
    FROM {SILVER_SCHEMA}.MOB_ORDER_ITEMS mi
    JOIN {SILVER_SCHEMA}.MOB_ORDERS mo ON mo.ORDER_ID = mi.ORDER_ID

    UNION ALL

    SELECT PRODUCT_SKU, PRODUCT_NAME, CATEGORY_CODE, UNIT_PRICE, ORDER_DATE AS SALE_DATE
    FROM {SILVER_SCHEMA}.WHL_ORDER_ITEMS wli
    JOIN {SILVER_SCHEMA}.WHL_ORDERS wo ON wo.ORDER_ID = wli.ORDER_ID

    UNION ALL

    SELECT PRODUCT_SKU, PRODUCT_NAME, CATEGORY_CODE, UNIT_PRICE, PURCHASE_DATE AS SALE_DATE
    FROM {SILVER_SCHEMA}.MKT_ORDERS
),
-- Pre-aggregate BRIDGE_CATEGORY: one canonical name per category code
-- Avoids unsupported correlated subquery in JOIN ON clause
cat_names AS (
    SELECT CATEGORY_CODE, MAX(CATEGORY_NAME) AS CATEGORY_NAME
    FROM {SILVER_SCHEMA}.BRIDGE_CATEGORY
    GROUP BY CATEGORY_CODE
),
-- Most frequent category per SKU
top_category AS (
    SELECT PRODUCT_SKU, CATEGORY_CODE,
           ROW_NUMBER() OVER (PARTITION BY PRODUCT_SKU ORDER BY COUNT(*) DESC) AS rn
    FROM all_items
    WHERE PRODUCT_SKU IS NOT NULL AND CATEGORY_CODE IS NOT NULL
    GROUP BY PRODUCT_SKU, CATEGORY_CODE
),
-- All categories per SKU (comma-separated canonical names)
all_cats AS (
    SELECT ai.PRODUCT_SKU,
           LISTAGG(DISTINCT bc.CATEGORY_NAME, ', ') AS ALL_CATEGORIES
    FROM all_items ai
    JOIN {SILVER_SCHEMA}.BRIDGE_CATEGORY bc ON bc.CATEGORY_CODE = ai.CATEGORY_CODE
    WHERE ai.PRODUCT_SKU IS NOT NULL
    GROUP BY ai.PRODUCT_SKU
),
-- Price aggregates + sale date range (plain GROUP BY — no nested window fns)
price_agg AS (
    SELECT
        PRODUCT_SKU,
        MAX(UNIT_PRICE)  AS CURRENT_UNIT_PRICE,
        AVG(UNIT_PRICE)  AS AVERAGE_UNIT_PRICE,
        MIN(UNIT_PRICE)  AS MIN_UNIT_PRICE,
        MAX(UNIT_PRICE)  AS MAX_UNIT_PRICE,
        MIN(SALE_DATE)   AS FIRST_SALE_DATE,
        MAX(SALE_DATE)   AS LAST_SALE_DATE
    FROM all_items
    WHERE PRODUCT_SKU IS NOT NULL
    GROUP BY PRODUCT_SKU
),
-- Latest product name per SKU using QUALIFY (avoids subquery wrapper)
latest_name AS (
    SELECT PRODUCT_SKU, PRODUCT_NAME
    FROM all_items
    WHERE PRODUCT_SKU IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY PRODUCT_SKU ORDER BY SALE_DATE DESC NULLS LAST) = 1
)
SELECT
    pa.PRODUCT_SKU,
    ln.PRODUCT_NAME,
    tc.CATEGORY_CODE                    AS PRIMARY_CATEGORY_CODE,
    cn.CATEGORY_NAME                    AS PRIMARY_CATEGORY_NAME,
    ac.ALL_CATEGORIES,
    pa.CURRENT_UNIT_PRICE,
    pa.AVERAGE_UNIT_PRICE,
    pa.MIN_UNIT_PRICE,
    pa.MAX_UNIT_PRICE,
    IFF(pa.LAST_SALE_DATE >= DATEADD(day, -180, CURRENT_DATE()), 'Active', 'Discontinued') AS PRODUCT_STATUS,
    pa.FIRST_SALE_DATE,
    pa.LAST_SALE_DATE
FROM price_agg    pa
JOIN latest_name  ln ON ln.PRODUCT_SKU   = pa.PRODUCT_SKU
JOIN top_category tc ON tc.PRODUCT_SKU   = pa.PRODUCT_SKU AND tc.rn = 1
LEFT JOIN cat_names cn ON cn.CATEGORY_CODE = tc.CATEGORY_CODE
LEFT JOIN all_cats  ac ON ac.PRODUCT_SKU   = pa.PRODUCT_SKU
'''

session.sql(f'CREATE OR REPLACE TEMP TABLE SILVER_PROD_SNAPSHOT AS {silver_products_sql}').collect()
prod_count = session.table('SILVER_PROD_SNAPSHOT').count()
print(f'Silver product snapshot: {prod_count:,} distinct SKUs')

In [ ]:
# ─────────────────────────────────────────────
# Cell 14 — DIM_PRODUCT  Step 2: Type 1 price update (overwrite in-place)
#            Step 3: Expire rows where PRODUCT_STATUS changed
#            Step 4: Insert new products + re-versioned rows
# ─────────────────────────────────────────────

# Type 1: overwrite prices on current rows (no history kept for prices)
session.sql(f'''
UPDATE {GOLD_SCHEMA}.DIM_PRODUCT tgt
SET
    CURRENT_UNIT_PRICE = src.CURRENT_UNIT_PRICE,
    AVERAGE_UNIT_PRICE = src.AVERAGE_UNIT_PRICE,
    MIN_UNIT_PRICE     = src.MIN_UNIT_PRICE,
    MAX_UNIT_PRICE     = src.MAX_UNIT_PRICE,
    LAST_SALE_DATE     = src.LAST_SALE_DATE
FROM SILVER_PROD_SNAPSHOT src
WHERE tgt.PRODUCT_SKU = src.PRODUCT_SKU
  AND tgt.IS_CURRENT  = TRUE
''').collect()

# SCD2: expire rows where PRODUCT_STATUS changed
session.sql(f'''
UPDATE {GOLD_SCHEMA}.DIM_PRODUCT tgt
SET
    VALID_TO   = CURRENT_DATE(),
    IS_CURRENT = FALSE
FROM SILVER_PROD_SNAPSHOT src
WHERE tgt.PRODUCT_SKU   = src.PRODUCT_SKU
  AND tgt.IS_CURRENT    = TRUE
  AND tgt.PRODUCT_STATUS != src.PRODUCT_STATUS
''').collect()

# Insert new products + new versions of just-expired products
next_key = get_next_key('DIM_PRODUCT', 'PRODUCT_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_PRODUCT (
    PRODUCT_KEY, PRODUCT_SKU, PRODUCT_NAME, PRIMARY_CATEGORY_CODE,
    PRIMARY_CATEGORY_NAME, ALL_CATEGORIES, CURRENT_UNIT_PRICE,
    AVERAGE_UNIT_PRICE, MIN_UNIT_PRICE, MAX_UNIT_PRICE,
    PRODUCT_STATUS, FIRST_SALE_DATE, LAST_SALE_DATE,
    EFFECTIVE_DATE, VALID_TO, IS_CURRENT
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY src.PRODUCT_SKU)  AS PRODUCT_KEY,
    src.PRODUCT_SKU,
    src.PRODUCT_NAME,
    src.PRIMARY_CATEGORY_CODE,
    src.PRIMARY_CATEGORY_NAME,
    src.ALL_CATEGORIES,
    src.CURRENT_UNIT_PRICE,
    src.AVERAGE_UNIT_PRICE,
    src.MIN_UNIT_PRICE,
    src.MAX_UNIT_PRICE,
    src.PRODUCT_STATUS,
    src.FIRST_SALE_DATE,
    src.LAST_SALE_DATE,
    CURRENT_DATE()  AS EFFECTIVE_DATE,
    NULL            AS VALID_TO,
    TRUE            AS IS_CURRENT
FROM SILVER_PROD_SNAPSHOT src
WHERE NOT EXISTS (
    SELECT 1 FROM {GOLD_SCHEMA}.DIM_PRODUCT g
    WHERE g.PRODUCT_SKU = src.PRODUCT_SKU AND g.IS_CURRENT = TRUE
)
''').collect()

cnt = validate('DIM_PRODUCT')
EXECUTION_STATS.append(('DIM_PRODUCT', cnt, 'SCD2'))

In [ ]:
# ─────────────────────────────────────────────
# Cell 15 — DIM_CAMPAIGN  (SCD Type 2)
# Source: SILVER.MKTG_CAMPAIGNS aggregated to campaign level
# SCD2 tracks: CAMPAIGN_NAME, CAMPAIGN_STATUS, END_DATE
# Depends on DIM_MARKETING_CHANNEL (loaded above)
# ─────────────────────────────────────────────

# Fix: CAMPAIGN_ID in Silver is a string (e.g. 'CMP0001').
# Snowflake cannot ALTER NUMBER → VARCHAR, so recreate the table with the correct type.
session.sql(f'DROP TABLE IF EXISTS {GOLD_SCHEMA}.DIM_CAMPAIGN').collect()
session.sql(f'''
CREATE TABLE {GOLD_SCHEMA}.DIM_CAMPAIGN (
    CAMPAIGN_KEY          NUMBER       NOT NULL,
    CAMPAIGN_ID           VARCHAR(50)  NOT NULL,
    CAMPAIGN_NAME         VARCHAR(200),
    MARKETING_CHANNEL_KEY NUMBER,
    TARGET_SEGMENT        VARCHAR(100),
    START_DATE            DATE,
    END_DATE              DATE,
    CAMPAIGN_STATUS       VARCHAR(50),
    EFFECTIVE_DATE        DATE         NOT NULL,
    VALID_TO              DATE,
    IS_CURRENT            BOOLEAN      NOT NULL DEFAULT TRUE
)
''').collect()
print('  DIM_CAMPAIGN recreated with CAMPAIGN_ID as VARCHAR(50)')

# Build campaign-level snapshot (aggregate daily rows → one row per campaign)
session.sql(f'''
CREATE OR REPLACE TEMP TABLE SILVER_CAMPAIGN_SNAPSHOT AS
WITH latest AS (
    SELECT
        mc.CAMPAIGN_ID::VARCHAR         AS CAMPAIGN_ID,
        mc.CAMPAIGN_NAME,
        mc.CAMPAIGN_STATUS,
        mc.CHANNEL,
        mc.TARGET_SEGMENT,
        mc.DATE
    FROM {SILVER_SCHEMA}.MKTG_CAMPAIGNS mc
    QUALIFY ROW_NUMBER() OVER (PARTITION BY mc.CAMPAIGN_ID ORDER BY mc.DATE DESC) = 1
),
agg AS (
    SELECT
        CAMPAIGN_ID::VARCHAR  AS CAMPAIGN_ID,
        MIN(START_DATE)       AS START_DATE,
        MAX(END_DATE)         AS END_DATE
    FROM {SILVER_SCHEMA}.MKTG_CAMPAIGNS
    GROUP BY CAMPAIGN_ID
)
SELECT
    l.CAMPAIGN_ID,
    l.CAMPAIGN_NAME,
    dmc.MARKETING_CHANNEL_KEY,
    l.TARGET_SEGMENT,
    a.START_DATE,
    a.END_DATE,
    l.CAMPAIGN_STATUS
FROM latest l
JOIN agg a ON a.CAMPAIGN_ID = l.CAMPAIGN_ID
LEFT JOIN {GOLD_SCHEMA}.DIM_MARKETING_CHANNEL dmc
    ON LOWER(dmc.MARKETING_CHANNEL_ID) = LOWER(l.CHANNEL)
''').collect()

snap_count = session.table('SILVER_CAMPAIGN_SNAPSHOT').count()
print(f'  Campaign snapshot: {snap_count:,} campaigns')

# No rows to expire on first run (table was just recreated)
# Insert all campaigns as new current rows
next_key = get_next_key('DIM_CAMPAIGN', 'CAMPAIGN_KEY')

session.sql(f'''
INSERT INTO {GOLD_SCHEMA}.DIM_CAMPAIGN (
    CAMPAIGN_KEY, CAMPAIGN_ID, CAMPAIGN_NAME, MARKETING_CHANNEL_KEY,
    TARGET_SEGMENT, START_DATE, END_DATE, CAMPAIGN_STATUS,
    EFFECTIVE_DATE, VALID_TO, IS_CURRENT
)
SELECT
    {next_key} - 1 + ROW_NUMBER() OVER (ORDER BY src.CAMPAIGN_ID)  AS CAMPAIGN_KEY,
    src.CAMPAIGN_ID,
    src.CAMPAIGN_NAME,
    src.MARKETING_CHANNEL_KEY,
    src.TARGET_SEGMENT,
    src.START_DATE,
    src.END_DATE,
    src.CAMPAIGN_STATUS,
    CURRENT_DATE()  AS EFFECTIVE_DATE,
    NULL            AS VALID_TO,
    TRUE            AS IS_CURRENT
FROM SILVER_CAMPAIGN_SNAPSHOT src
''').collect()

cnt = validate('DIM_CAMPAIGN')
EXECUTION_STATS.append(('DIM_CAMPAIGN', cnt, 'SCD2'))

---
## Validation Summary

In [ ]:
# ─────────────────────────────────────────────
# Cell 16 — Dimension Load Summary
# ─────────────────────────────────────────────
print('='*65)
print(f'  Gold Dimension Load Summary — {RUN_DATE}')
print('='*65)
print(f'  {"Table":<35} {"Rows":>8}  {"Strategy"}')
print('-'*65)
for table, rows, strategy in EXECUTION_STATS:
    print(f'  {table:<35} {rows:>8,}  {strategy}')

print('-'*65)
# DIM_DATE and DIM_CHANNEL are static — show their counts too
dd   = count_table('DIM_DATE')
dch  = count_table('DIM_CHANNEL')
print(f'  {"DIM_DATE (pre-seeded)":<35} {dd:>8,}  static')
print(f'  {"DIM_CHANNEL (pre-seeded)":<35} {dch:>8,}  static')
print('='*65)
print('  ✓ All dimensions loaded. Run GOLD_INCREMENTAL_LOAD_FACTS.ipynb next.')
print('='*65)